# PyMC-NMR Tutorial 1: STA30 Basin Hopping

This example demonstrates how to run the STA30 example with the modern YAML configuration.

First, obtain an extended xyz file of the structure (e.g. from the ICSD) or use the provided `cif77_opt.xyz`.

In [ ]:
from ase import Atoms
from ase.io import read, write
import numpy as np

def _substitute_al(symbols):
    slist = [i for i, s in enumerate(symbols) if s == 'Si']
    for _ in range(10):
        choice = slist.pop(int(len(slist) * np.random.random()))
        symbols[choice] = 'Al'

atoms_object = read('cif77_opt.xyz')
_substitute_al(atoms_object.symbols)
write(filename='basin.xyz', images=atoms_object, format='extxyz')
print('Created basin.xyz')

### Write the YAML configuration

PyMC-NMR now uses a single `config.yaml` file. Species are derived automatically from the structure, so you no longer need a separate `control` or `potentials` file.

In [ ]:
import yaml

config_data = {
    'job': {
        'structure_path': 'basin.xyz',
        'structure_method': 'basinhop',
        'temperature': 1300.0,
        'equil_steps': 2
    },
    'potential': {
        'arch': 'mace_mp',
        'model': 'MACE-matpes-r2scan-omat-ft.model',
        'device': 'cpu',
        'precision': 'float64'
    },
    'monte': {
        'steps': 5
    },
    'moves': {
        'swaps': [
            {'pairs': [['Al', 'Si']], 'frequency': 100}
        ]
    },
    'relax': {
        'method': 'bfgs',
        'steps': 500,
        'tol': 0.01
    }
}

with open('config.yaml', 'w') as f:
    yaml.safe_dump(config_data, f, default_flow_style=False)
print('Written config.yaml')

### Run the simulation

Import `run_simulation` from the source tree and execute the basin-hopping search.

In [ ]:
from pymc_nmr.monte import run_simulation

run_simulation(config_path='config.yaml', structure_path='basin.xyz', log_path='mc.log')


### Output files

After the run you will find:
* **mc.log**: calculation parameters, accepted swaps and energies.
* **restart.xyz**: final optimised structure.
* **archive.xyz**: structural trajectory history.